# Phase 1A: Reach-Avoid vs ELBO Ranking Analysis (Cancer)

Zero-retraining experiments on existing VCIP models (Cancer, 4 gammas × 5 seeds).

**Experiments:**
- E1: Ranking comparison — RA vs ELBO Top-1 agreement on Cancer ground truth
- E2: Margin analysis — pairwise margin distributions (validates T2 theorem)
- E3: Cross-seed ranking stability (Cancer only; MIMIC deferred to Phase 1B)

**Decision gate:** If RA scoring does not substantially improve Top-1 agreement over ELBO on Cancer ground truth, reassess before Phase 2.

**Prerequisites:** Run `scripts/phase1/run_phase1.sh` on Vast.ai to generate augmented case_infos with RA scores across all gammas.

In [1]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, rankdata
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.2)
%matplotlib inline

## Load Data

In [ ]:
# Path to results — adjust if results are in a different location
# After running run_phase1.sh on Vast.ai, download case_infos to results_remote/phase1_ra/
RESULTS_BASE = Path('../../results_remote/phase1_ra/my_outputs/cancer_sim_cont/22')
# Fallback: if results were placed in the original location
RESULTS_BASE_FALLBACK = Path('../../results_remote/r/my_outputs/cancer_sim_cont/22')

SEEDS = [10, 101, 1010, 10101, 101010]
GAMMAS = [1, 2, 3, 4]
TAUS = [2, 4, 6, 8]
K = 100  # number of candidate sequences per individual

# Load all case_infos: {gamma: {seed: {tau: [case_infos]}}}
all_data = {}
for gamma in GAMMAS:
    all_data[gamma] = {}
    for seed in SEEDS:
        for base in [RESULTS_BASE, RESULTS_BASE_FALLBACK]:
            pkl_path = base / f'coeff_{gamma}' / 'VCIP' / 'train' / 'True' / 'case_infos' / str(seed) / 'False' / 'case_infos_VCIP.pkl'
            if pkl_path.exists():
                with open(pkl_path, 'rb') as f:
                    data = pickle.load(f)
                all_data[gamma][seed] = data['VCIP']
                break
        else:
            print(f'Missing: gamma={gamma}, seed={seed}')

# Summary
for gamma in GAMMAS:
    n_seeds = len(all_data[gamma])
    if n_seeds > 0:
        sample_seed = list(all_data[gamma].keys())[0]
        taus_found = list(all_data[gamma][sample_seed].keys())
        n_ind = len(all_data[gamma][sample_seed][taus_found[0]])
        print(f'gamma={gamma}: {n_seeds} seeds, taus={taus_found}, {n_ind} individuals/tau')
    else:
        print(f'gamma={gamma}: NO DATA')

# Check if RA scores are present (from gamma=4, seed=10)
sample_gamma = max(g for g in GAMMAS if all_data[g])
sample_seed = list(all_data[sample_gamma].keys())[0]
sample_tau = list(all_data[sample_gamma][sample_seed].keys())[0]
sample_ci = all_data[sample_gamma][sample_seed][sample_tau][0]
has_ra = 'true_ra_scores' in sample_ci
print(f'\nRA scores present: {has_ra}')
print(f'Case info keys: {list(sample_ci.keys())}')

## E1: Ranking Comparison — RA vs ELBO

For each individual, compare which scoring method better identifies the true best sequence.

**Metrics:**
- Top-1 agreement: does the best-scoring sequence also have the best ground-truth outcome?
- GRP (Ground-truth Rank Percentile): rank of true sequence / k
- Spearman correlation between scoring and ground-truth ranking

In [ ]:
def compute_ranking_metrics(case_infos, has_ra=True):
    """Compute ranking quality metrics for ELBO and RA scoring."""
    results = []
    for ci in case_infos:
        k = len(ci['model_losses'])
        
        # Ground truth: lower true_loss = better
        true_ranks = rankdata(ci['true_losses'])  # rank 1 = lowest loss = best
        best_true_idx = np.argmin(ci['true_losses'])
        
        # ELBO ranking: lower ELBO = better (ELBO is negative, more negative = worse fit)
        # Actually in VCIP, higher ELBO = better fit, so best = argmax
        # But true_losses uses RMSE (lower = better), so correlation should be negative
        # Let's use the raw values and compute metrics
        elbo_ranks = rankdata(-ci['model_losses'])  # negate: higher ELBO gets rank 1
        best_elbo_idx = np.argmax(ci['model_losses'])
        
        # ELBO Top-1: does ELBO's best match ground truth's best?
        elbo_top1 = int(best_elbo_idx == best_true_idx)
        
        # ELBO-True Spearman (negative correlation expected: high ELBO ↔ low true_loss)
        elbo_true_rho, _ = spearmanr(ci['model_losses'], ci['true_losses'])
        
        # GRP: rank percentile of true sequence (last in all_sequences)
        elbo_grp = 1 - (ci['true_sequence_rank'] - 1) / k
        
        row = {
            'individual_id': ci['individual_id'],
            'elbo_top1': elbo_top1,
            'elbo_true_rho': elbo_true_rho,
            'elbo_grp': elbo_grp,
            'elbo_best_true_rank': true_ranks[best_elbo_idx],  # ground-truth rank of ELBO's pick
        }
        
        if has_ra:
            ra_scores = ci['true_ra_scores']
            # RA ranking: higher RA score = better
            best_ra_idx = np.argmax(ra_scores)
            ra_top1 = int(best_ra_idx == best_true_idx)
            ra_true_rho, _ = spearmanr(ra_scores, -ci['true_losses'])  # should be positive
            ra_grp = 1 - (ci.get('true_sequence_ra_rank', k) - 1) / k
            
            # Also: does RA's best sequence actually reach the target?
            row.update({
                'ra_top1': ra_top1,
                'ra_true_rho': ra_true_rho,
                'ra_grp': ra_grp,
                'ra_best_true_rank': true_ranks[best_ra_idx],
                'ra_best_ra_score': ra_scores[best_ra_idx],
                'elbo_best_ra_score': ra_scores[best_elbo_idx],
            })
        
        results.append(row)
    return results

In [ ]:
# Compute metrics across all gammas, seeds, and taus
import pandas as pd

all_metrics = []
for gamma in GAMMAS:
    for seed in all_data[gamma]:
        for tau in TAUS:
            if tau not in all_data[gamma][seed]:
                continue
            metrics = compute_ranking_metrics(all_data[gamma][seed][tau], has_ra=has_ra)
            for m in metrics:
                m['seed'] = seed
                m['tau'] = tau
                m['gamma'] = gamma
            all_metrics.extend(metrics)

df = pd.DataFrame(all_metrics)
n_gammas = len([g for g in GAMMAS if all_data[g]])
print(f'Total observations: {len(df)} ({n_gammas} gammas × {len(SEEDS)} seeds × {len(TAUS)} taus × ~100 individuals)')
df.head()

In [ ]:
# E1 Summary Table: Mean metrics by gamma × tau (averaged across seeds)
if has_ra:
    summary_cols = ['elbo_top1', 'ra_top1', 'elbo_true_rho', 'ra_true_rho', 
                    'elbo_best_true_rank', 'ra_best_true_rank']
else:
    summary_cols = ['elbo_top1', 'elbo_true_rho', 'elbo_best_true_rank']

# Aggregate per seed×tau×gamma, then mean/std across seeds
per_seed = df.groupby(['gamma', 'seed', 'tau'])[summary_cols].mean().reset_index()

print('=== E1: Ranking Quality by Gamma × Tau (mean ± std across 5 seeds) ===')
print()

for gamma in GAMMAS:
    if gamma not in per_seed.gamma.values:
        continue
    print(f'--- gamma={gamma} ---')
    sub = per_seed[per_seed.gamma == gamma]
    summary = sub.groupby('tau')[summary_cols].agg(['mean', 'std'])
    
    key_cols = ['elbo_top1', 'ra_top1'] if has_ra else ['elbo_top1']
    for col in key_cols:
        for tau in TAUS:
            if tau in summary.index:
                m = summary.loc[tau, (col, 'mean')]
                s = summary.loc[tau, (col, 'std')]
                print(f'  {col} tau={tau}: {m:.3f} ± {s:.3f}')
    print()

# Overall summary across all gammas
print('--- ALL GAMMAS (pooled) ---')
overall = per_seed.groupby('tau')[summary_cols].agg(['mean', 'std'])
for col in ['elbo_top1'] + (['ra_top1'] if has_ra else []):
    for tau in TAUS:
        m = overall.loc[tau, (col, 'mean')]
        s = overall.loc[tau, (col, 'std')]
        print(f'  {col} tau={tau}: {m:.3f} ± {s:.3f}')

In [ ]:
# E1 Visualization: Top-1 agreement comparison — gamma=4 (primary) + all gammas heatmap
if has_ra:
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    
    # Panel 1: Top-1 agreement for gamma=4 (decision gate)
    ax = axes[0]
    g4 = per_seed[per_seed.gamma == 4]
    top1_data = g4.groupby('tau')[['elbo_top1', 'ra_top1']].agg(['mean', 'std'])
    x = np.arange(len(TAUS))
    w = 0.35
    ax.bar(x - w/2, [top1_data.loc[t, ('elbo_top1', 'mean')] for t in TAUS],
           yerr=[top1_data.loc[t, ('elbo_top1', 'std')] for t in TAUS],
           width=w, label='ELBO', color='steelblue', capsize=3)
    ax.bar(x + w/2, [top1_data.loc[t, ('ra_top1', 'mean')] for t in TAUS],
           yerr=[top1_data.loc[t, ('ra_top1', 'std')] for t in TAUS],
           width=w, label='RA (ground truth)', color='coral', capsize=3)
    ax.set_xticks(x)
    ax.set_xticklabels([f'τ={t}' for t in TAUS])
    ax.set_ylabel('Top-1 Agreement Rate')
    ax.set_title('E1: Top-1 Agreement (γ=4)')
    ax.legend()
    ax.set_ylim(0, 1)
    
    # Panel 2: Heatmap of RA Top-1 improvement (RA - ELBO) across gamma × tau
    ax = axes[1]
    improvement = per_seed.groupby(['gamma', 'tau']).apply(
        lambda x: x['ra_top1'].mean() - x['elbo_top1'].mean()
    ).unstack()
    sns.heatmap(improvement, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
                ax=ax, xticklabels=[f'τ={t}' for t in TAUS],
                yticklabels=[f'γ={g}' for g in GAMMAS])
    ax.set_title('E1: RA Top-1 Improvement (RA − ELBO)')
    
    # Panel 3: Best-pick ground-truth rank for gamma=4
    ax = axes[2]
    rank_data = g4.groupby('tau')[['elbo_best_true_rank', 'ra_best_true_rank']].agg(['mean', 'std'])
    ax.bar(x - w/2, [rank_data.loc[t, ('elbo_best_true_rank', 'mean')] for t in TAUS],
           yerr=[rank_data.loc[t, ('elbo_best_true_rank', 'std')] for t in TAUS],
           width=w, label='ELBO', color='steelblue', capsize=3)
    ax.bar(x + w/2, [rank_data.loc[t, ('ra_best_true_rank', 'mean')] for t in TAUS],
           yerr=[rank_data.loc[t, ('ra_best_true_rank', 'std')] for t in TAUS],
           width=w, label='RA (ground truth)', color='coral', capsize=3)
    ax.set_xticks(x)
    ax.set_xticklabels([f'τ={t}' for t in TAUS])
    ax.set_ylabel('Ground-Truth Rank of Best Pick (lower=better)')
    ax.set_title('E1: Quality of Best-Ranked Sequence (γ=4)')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig('../phase1/e1_ranking_comparison.pdf', bbox_inches='tight')
    plt.show()
else:
    print('RA scores not available. Run scripts/phase1/run_phase1.sh first.')

## E2: Margin Analysis (T2 Theorem Validation)

T2 states that RA ranking is preserved when the margin `m = |P(ā₁) - P(ā₂)| > 2ε_VI + 2τ·ε_soft(κ)`.

We compute pairwise margin distributions under ELBO and RA scoring, and measure the fraction of pairs where the margin is "large enough" to resist ranking inversions.

In [ ]:
def compute_pairwise_margins(case_infos, n_pairs_per_individual=500):
    """Compute pairwise margin distributions for ELBO and RA scoring.
    
    For each individual, sample random pairs of sequences and compute
    the margin (score difference) under each scoring method.
    Also track whether the ranking agrees with ground truth.
    """
    elbo_margins = []
    ra_margins = []
    elbo_correct = []  # does ELBO margin sign match ground-truth?
    ra_correct = []    # does RA margin sign match ground-truth?
    
    rng = np.random.RandomState(42)
    
    for ci in case_infos:
        k = len(ci['model_losses'])
        losses = ci['true_losses']
        elbos = ci['model_losses']
        
        # Sample random pairs
        idx_a = rng.randint(0, k, n_pairs_per_individual)
        idx_b = rng.randint(0, k, n_pairs_per_individual)
        mask = idx_a != idx_b
        idx_a, idx_b = idx_a[mask], idx_b[mask]
        
        # Ground truth: a is better if true_loss[a] < true_loss[b]
        gt_sign = np.sign(losses[idx_b] - losses[idx_a])  # +1 if a is better
        
        # ELBO margin: higher ELBO = better, so a is better if elbo[a] > elbo[b]
        elbo_diff = elbos[idx_a] - elbos[idx_b]
        elbo_margins.extend(np.abs(elbo_diff).tolist())
        elbo_correct.extend((np.sign(elbo_diff) == gt_sign).astype(int).tolist())
        
        if 'true_ra_scores' in ci:
            ra = ci['true_ra_scores']
            ra_diff = ra[idx_a] - ra[idx_b]
            ra_margins.extend(np.abs(ra_diff).tolist())
            # RA: higher = better, same sign convention as gt_sign
            ra_correct.extend((np.sign(ra_diff) == gt_sign).astype(int).tolist())
    
    return {
        'elbo_margins': np.array(elbo_margins),
        'elbo_correct': np.array(elbo_correct),
        'ra_margins': np.array(ra_margins) if ra_margins else None,
        'ra_correct': np.array(ra_correct) if ra_correct else None,
    }

In [ ]:
# Compute margins for each gamma × tau
margin_results = {}  # {(gamma, tau): margin_dict}
for gamma in GAMMAS:
    for tau in TAUS:
        # Pool all seeds for this gamma × tau
        all_cis = []
        for seed in all_data[gamma]:
            if tau in all_data[gamma][seed]:
                all_cis.extend(all_data[gamma][seed][tau])
        if not all_cis:
            continue
        margin_results[(gamma, tau)] = compute_pairwise_margins(all_cis)

# Print summary for gamma=4 (primary)
print('=== E2: Pairwise Ranking Accuracy (gamma=4) ===')
for tau in TAUS:
    key = (4, tau)
    if key not in margin_results:
        continue
    em = margin_results[key]['elbo_margins']
    ec = margin_results[key]['elbo_correct']
    print(f'tau={tau}: ELBO accuracy={ec.mean():.3f}, median margin={np.median(em):.4f}')
    
    if margin_results[key]['ra_margins'] is not None:
        rm = margin_results[key]['ra_margins']
        rc = margin_results[key]['ra_correct']
        print(f'        RA accuracy={rc.mean():.3f}, median margin={np.median(rm):.4f}')

# Cross-gamma summary
if has_ra:
    print('\n=== E2: RA vs ELBO Pairwise Accuracy Across Gammas ===')
    for gamma in GAMMAS:
        for tau in TAUS:
            key = (gamma, tau)
            if key not in margin_results or margin_results[key]['ra_margins'] is None:
                continue
            ec = margin_results[key]['elbo_correct'].mean()
            rc = margin_results[key]['ra_correct'].mean()
            print(f'  gamma={gamma}, tau={tau}: ELBO={ec:.3f}, RA={rc:.3f}, Δ={rc-ec:+.3f}')

In [ ]:
# E2 Visualization: Margin distributions and accuracy vs margin (gamma=4)
if has_ra:
    fig, axes = plt.subplots(2, len(TAUS), figsize=(5*len(TAUS), 8))
    
    for j, tau in enumerate(TAUS):
        key = (4, tau)
        if key not in margin_results:
            continue
        mr = margin_results[key]
        
        # Top row: margin distributions
        ax = axes[0, j]
        ax.hist(mr['elbo_margins'], bins=50, alpha=0.5, label='ELBO', color='steelblue', density=True)
        if mr['ra_margins'] is not None:
            ax.hist(mr['ra_margins'], bins=50, alpha=0.5, label='RA', color='coral', density=True)
        ax.set_title(f'γ=4, τ={tau}: Margin Distribution')
        ax.set_xlabel('|margin|')
        ax.set_ylabel('Density')
        ax.legend()
        
        # Bottom row: accuracy vs margin bin
        ax = axes[1, j]
        items = [('ELBO', mr['elbo_margins'], mr['elbo_correct'], 'steelblue')]
        if mr['ra_margins'] is not None:
            items.append(('RA', mr['ra_margins'], mr['ra_correct'], 'coral'))
        
        for name, margins, correct, color in items:
            percentiles = np.percentile(margins, np.linspace(0, 100, 11))
            bin_centers = []
            bin_accs = []
            for i in range(len(percentiles)-1):
                mask = (margins >= percentiles[i]) & (margins < percentiles[i+1])
                if mask.sum() > 0:
                    bin_centers.append((percentiles[i] + percentiles[i+1]) / 2)
                    bin_accs.append(correct[mask].mean())
            ax.plot(bin_centers, bin_accs, 'o-', label=name, color=color)
        ax.axhline(0.5, ls='--', color='gray', alpha=0.5)
        ax.set_title(f'γ=4, τ={tau}: Ranking Accuracy vs Margin')
        ax.set_xlabel('Margin')
        ax.set_ylabel('Pairwise Ranking Accuracy')
        ax.legend()
        ax.set_ylim(0.3, 1.0)
    
    plt.tight_layout()
    plt.savefig('../phase1/e2_margin_analysis.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# E2 Key metric: fraction of pairs where ranking is preserved at various margin thresholds
# This directly validates T2: larger margins → more robust rankings
if has_ra:
    print('=== E2: Pairwise Ranking Accuracy by Margin Percentile (gamma=4) ===')
    print()
    for tau in TAUS:
        key = (4, tau)
        if key not in margin_results or margin_results[key]['ra_margins'] is None:
            continue
        mr = margin_results[key]
        print(f'tau={tau}:')
        for pct in [25, 50, 75, 90]:
            elbo_thresh = np.percentile(mr['elbo_margins'], pct)
            ra_thresh = np.percentile(mr['ra_margins'], pct)
            
            elbo_mask = mr['elbo_margins'] >= elbo_thresh
            ra_mask = mr['ra_margins'] >= ra_thresh
            
            elbo_acc = mr['elbo_correct'][elbo_mask].mean()
            ra_acc = mr['ra_correct'][ra_mask].mean()
            
            print(f'  Top {100-pct}% margin: ELBO acc={elbo_acc:.3f}, RA acc={ra_acc:.3f}')
        print()
    
    # Cross-gamma: show the fraction of pairs where RA margin exceeds ELBO margin
    print('=== E2: Fraction of Pairs with RA Margin > ELBO Margin ===')
    for gamma in GAMMAS:
        for tau in TAUS:
            key = (gamma, tau)
            if key not in margin_results or margin_results[key]['ra_margins'] is None:
                continue
            mr = margin_results[key]
            # Compare at same percentile thresholds
            ra_median = np.median(mr['ra_margins'])
            elbo_median = np.median(mr['elbo_margins'])
            print(f'  gamma={gamma}, tau={tau}: RA median={ra_median:.4f}, ELBO median={elbo_median:.4f}')

## E3: Cross-Seed Ranking Stability (Cancer)

For each individual × gamma × tau, compare how stable the rankings are across seeds.
More stable rankings → more reliable clinical recommendations.

Note: MIMIC stability analysis deferred to Phase 1B.

In [ ]:
# Cross-seed stability: for each individual, compute rank std across seeds
stability_results = []

for gamma in GAMMAS:
    for tau in TAUS:
        individual_elbo_ranks = {}
        individual_ra_ranks = {}
        
        for seed in all_data[gamma]:
            if tau not in all_data[gamma][seed]:
                continue
            for ci in all_data[gamma][seed][tau]:
                iid = ci['individual_id']
                
                if iid not in individual_elbo_ranks:
                    individual_elbo_ranks[iid] = []
                    individual_ra_ranks[iid] = []
                
                individual_elbo_ranks[iid].append(ci['true_sequence_rank'])
                
                if has_ra:
                    individual_ra_ranks[iid].append(ci.get('true_sequence_ra_rank', K))
        
        for iid in individual_elbo_ranks:
            if len(individual_elbo_ranks[iid]) < 2:
                continue
            elbo_std = np.std(individual_elbo_ranks[iid])
            row = {'gamma': gamma, 'tau': tau, 'individual_id': iid, 'elbo_rank_std': elbo_std}
            
            if has_ra and individual_ra_ranks.get(iid) and len(individual_ra_ranks[iid]) >= 2:
                row['ra_rank_std'] = np.std(individual_ra_ranks[iid])
            
            stability_results.append(row)

df_stab = pd.DataFrame(stability_results)

print('=== E3: Cross-Seed Ranking Stability (rank std, lower=more stable) ===')
print()
for gamma in GAMMAS:
    sub = df_stab[df_stab.gamma == gamma]
    if sub.empty:
        continue
    print(f'--- gamma={gamma} ---')
    stab_cols = ['elbo_rank_std']
    if has_ra and 'ra_rank_std' in sub.columns:
        stab_cols.append('ra_rank_std')
    stab_summary = sub.groupby('tau')[stab_cols].agg(['mean', 'median'])
    print(stab_summary)
    print()

In [ ]:
# E3 Visualization: stability comparison (gamma=4 primary + cross-gamma heatmap)
if has_ra and 'ra_rank_std' in df_stab.columns:
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    
    # Panel 1: Box plot of rank std by tau for gamma=4
    ax = axes[0]
    g4_stab = df_stab[df_stab.gamma == 4]
    positions_elbo = np.arange(len(TAUS)) * 2
    positions_ra = positions_elbo + 0.7
    
    bp1 = ax.boxplot([g4_stab[g4_stab.tau==t]['elbo_rank_std'].dropna().values for t in TAUS],
                     positions=positions_elbo, widths=0.5, patch_artist=True)
    bp2 = ax.boxplot([g4_stab[g4_stab.tau==t]['ra_rank_std'].dropna().values for t in TAUS],
                     positions=positions_ra, widths=0.5, patch_artist=True)
    
    for patch in bp1['boxes']:
        patch.set_facecolor('steelblue')
        patch.set_alpha(0.7)
    for patch in bp2['boxes']:
        patch.set_facecolor('coral')
        patch.set_alpha(0.7)
    
    ax.set_xticks(positions_elbo + 0.35)
    ax.set_xticklabels([f'τ={t}' for t in TAUS])
    ax.set_ylabel('Rank Std Across Seeds')
    ax.set_title('E3: Cross-Seed Stability (γ=4)')
    ax.legend([bp1['boxes'][0], bp2['boxes'][0]], ['ELBO', 'RA'])
    
    # Panel 2: Scatter — ELBO rank std vs RA rank std per individual (all gammas)
    ax = axes[1]
    for gamma in GAMMAS:
        sub = df_stab[df_stab.gamma == gamma].dropna(subset=['ra_rank_std'])
        ax.scatter(sub['elbo_rank_std'], sub['ra_rank_std'], alpha=0.15, s=8, label=f'γ={gamma}')
    max_val = max(df_stab['elbo_rank_std'].max(), df_stab['ra_rank_std'].dropna().max())
    ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.3)
    ax.set_xlabel('ELBO Rank Std')
    ax.set_ylabel('RA Rank Std')
    ax.set_title('E3: Per-Individual Stability (below diagonal = RA more stable)')
    ax.legend()
    
    # Panel 3: Heatmap of mean stability improvement (ELBO std - RA std) across gamma × tau
    ax = axes[2]
    stab_improvement = df_stab.dropna(subset=['ra_rank_std']).groupby(['gamma', 'tau']).apply(
        lambda x: x['elbo_rank_std'].mean() - x['ra_rank_std'].mean()
    ).unstack()
    sns.heatmap(stab_improvement, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
                ax=ax, xticklabels=[f'τ={t}' for t in TAUS],
                yticklabels=[f'γ={g}' for g in GAMMAS])
    ax.set_title('E3: Stability Improvement (ELBO std − RA std, >0 = RA better)')
    
    plt.tight_layout()
    plt.savefig('../phase1/e3_ranking_stability.pdf', bbox_inches='tight')
    plt.show()

## Decision Gate Summary

Evaluate whether to proceed with Phase 2 based on Phase 1 results.

In [ ]:
print('=' * 70)
print('PHASE 1A DECISION GATE SUMMARY')
print('=' * 70)
print()

if has_ra:
    # Primary decision: gamma=4 (strongest confounding, most challenging)
    print('--- Primary (gamma=4) ---')
    g4_seed = per_seed[per_seed.gamma == 4]
    for tau in TAUS:
        sub = g4_seed[g4_seed.tau == tau]
        if sub.empty:
            continue
        elbo_t1 = sub['elbo_top1'].mean()
        ra_t1 = sub['ra_top1'].mean()
        improvement = ra_t1 - elbo_t1
        
        stab_sub = df_stab[(df_stab.gamma == 4) & (df_stab.tau == tau)]
        elbo_stab = stab_sub['elbo_rank_std'].mean()
        ra_stab = stab_sub['ra_rank_std'].mean() if 'ra_rank_std' in stab_sub.columns else float('nan')
        
        status = 'PASS' if improvement > 0.05 else ('MARGINAL' if improvement > 0 else 'FAIL')
        print(f'  tau={tau}: ELBO Top-1={elbo_t1:.3f} → RA Top-1={ra_t1:.3f} '
              f'(Δ={improvement:+.3f}) [{status}]')
        print(f'           Stability: ELBO std={elbo_stab:.1f} → RA std={ra_stab:.1f}')
    
    # Cross-gamma summary
    print()
    print('--- Cross-Gamma Summary ---')
    for gamma in GAMMAS:
        sub = per_seed[per_seed.gamma == gamma]
        if sub.empty:
            continue
        elbo_t1 = sub['elbo_top1'].mean()
        ra_t1 = sub['ra_top1'].mean()
        print(f'  gamma={gamma}: ELBO Top-1={elbo_t1:.3f}, RA Top-1={ra_t1:.3f}, Δ={ra_t1-elbo_t1:+.3f}')
    
    # Overall
    print()
    overall_elbo = per_seed['elbo_top1'].mean()
    overall_ra = per_seed['ra_top1'].mean()
    print(f'Overall: ELBO Top-1={overall_elbo:.3f}, RA Top-1={overall_ra:.3f}, Δ={overall_ra-overall_elbo:+.3f}')
    print()
    
    # Decision
    g4_elbo = g4_seed['elbo_top1'].mean()
    g4_ra = g4_seed['ra_top1'].mean()
    
    if g4_ra > g4_elbo + 0.05:
        print('DECISION: PROCEED to Phase 1B (MIMIC) and Phase 2 (RA-aware retraining).')
        print('RA scoring shows substantial improvement on Cancer ground truth.')
    elif g4_ra > g4_elbo:
        print('DECISION: PROCEED with CAUTION to Phase 1B.')
        print('RA shows modest improvement. Investigate whether stability or margin gains')
        print('justify the additional complexity before committing to Phase 2.')
    else:
        print('DECISION: REASSESS approach.')
        print('RA scoring does not improve over ELBO on Cancer ground truth.')
        print('Consider: (1) tuning T/S thresholds, (2) alternative target definitions,')
        print('(3) pivoting to a different contribution.')
else:
    print('RA scores not available. Run scripts/phase1/run_phase1.sh on Vast.ai first.')
    print()
    print('Current ELBO-only baseline metrics (gamma=4):')
    g4_seed = per_seed[per_seed.gamma == 4]
    for tau in TAUS:
        sub = g4_seed[g4_seed.tau == tau]
        if not sub.empty:
            print(f'  tau={tau}: ELBO Top-1={sub["elbo_top1"].mean():.3f} ± {sub["elbo_top1"].std():.3f}')